In [1]:
import biogeme.database as db
import geopandas as gpd
import pandas as pd
from biogeme import models
from biogeme.biogeme import BIOGEME
from biogeme.expressions import Beta, Variable

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.
c:\Users\lain\Documents\housing_choice\.venv\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(
c:\Users\lain\Documents\housing_choice\.venv\Lib\site-packages\tqdm_joblib\__init__.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


* Costo
* Superficie
* Parques
* Distancia a la frontera
* Baldíos y parques a 5 minutos

In [98]:
df_transactions = pd.read_parquet("./transactions.parquet")

unused_frac = df_transactions["fraccionamiento"].value_counts()
unused_frac = unused_frac[unused_frac < 10].index
df_transactions = df_transactions.loc[
    lambda df: ~df["fraccionamiento"].isin(unused_frac)
]

cat = df_transactions["fraccionamiento"].astype("category")
label_to_frac_map = dict(enumerate(cat.cat.categories))
frac_to_label_map = {v: k for k, v in label_to_frac_map.items()}

price_per_sqm = (
    df_transactions.assign(
        price_per_sqm=lambda df: df["valor_operacion"] / df["mts_superficie"]
    )
    .groupby("fraccionamiento")["price_per_sqm"]
    .median()
)

df_transactions = df_transactions[["fraccionamiento"]].assign(
    fraccionamiento=lambda df: df["fraccionamiento"].map(frac_to_label_map)
)

In [100]:
df_frac_features = (
    gpd.read_parquet("./frac_features.geoparquet")
    .drop(columns=["geometry"])
    .join(price_per_sqm)
)

df_frac_features = (
    df_frac_features.reset_index(names="fraccionamiento")
    .assign(fraccionamiento=lambda df: df["fraccionamiento"].map(frac_to_label_map))
    .dropna(subset=["fraccionamiento"])
    .assign(fraccionamiento=lambda df: df["fraccionamiento"].astype(int))
    .set_index("fraccionamiento")
    .drop(columns=["tree_coverage_frac", "urbanized_area_frac"])
)

for c in df_frac_features.columns:
    df_frac_features[c] = df_frac_features[c].fillna(df_frac_features[c].mean())

In [102]:
feature_dict = {}
for col in df_frac_features.columns:
    for idx, val in df_frac_features[col].items():
        feature_dict[f"{col}_{idx}"] = val

for col_name, value in feature_dict.items():
    df_transactions[col_name] = value

df_transactions = df_transactions.copy()  # Defragment DataFrame

C:\Users\lain\AppData\Local\Temp\ipykernel_6636\623320871.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transactions[col_name] = value
C:\Users\lain\AppData\Local\Temp\ipykernel_6636\623320871.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transactions[col_name] = value
C:\Users\lain\AppData\Local\Temp\ipykernel_6636\623320871.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

In [103]:
database = db.Database("housing_choice_model", df_transactions)

choice = Variable("fraccionamiento")

betas = {
    f"beta_{col}": Beta(f"beta_{col}", 0, None, None, 0)
    for col in df_frac_features.columns
}

V = {}
av = {}
for i in range(len(df_frac_features)):
    var_map = {
        f"var_{col}_{i}": Variable(f"{col}_{i}") for col in df_frac_features.columns
    }
    V[i] = sum(
        betas[f"beta_{col}"] * var_map[f"var_{col}_{i}"]
        for col in df_frac_features.columns
    )
    av[i] = 1

logprob = models.loglogit(V, av, choice)
biogeme_model = BIOGEME(
    database,
    logprob,
    parameters="./params/test_1.yaml",
    generate_yaml=False,
    generate_html=True,
    save_iterations=False,
)
biogeme_model.model_name = "test_1"

results = biogeme_model.estimate()

In [104]:
print(results.short_summary())

Results for model test_1
Nbr of parameters:		8
Sample size:			7106
Excluded data:			0
Final log likelihood:		-19838.01
Akaike Information Criterion:	39692.01
Bayesian Information Criterion:	39746.96



In [105]:
print(results.get_estimated_parameters())

                         Name     Value  Robust std err.  Robust t-stat.  \
0       beta_jobs_manufacture  0.000398         0.000027       14.736962   
1    beta_jobs_infrastructure -0.005047         0.000246      -20.537662   
2       beta_jobs_specialized  0.005687         0.000323       17.632221   
3              beta_jobs_care -0.000995         0.000070      -14.301243   
4          beta_accessibility  6.241459         0.299795       20.819085   
5        beta_accessibility_0  2.902390         1.304925        2.224182   
6  beta_travel_time_to_center  0.016788         0.000593       28.300140   
7          beta_price_per_sqm -0.000063         0.000013       -5.026695   

   Robust p-value  
0    0.000000e+00  
1    0.000000e+00  
2    0.000000e+00  
3    0.000000e+00  
4    0.000000e+00  
5    2.613620e-02  
6    0.000000e+00  
7    4.990040e-07  


C:\Users\lain\AppData\Local\Temp\ipykernel_6636\2081322227.py:1: DeprecationWarning: get_estimated_parameters is deprecated. Use get_pandas_estimated_parameters(estimation_results=my_results) instead
  print(results.get_estimated_parameters())
